### DAY 5: LLM BEHAVIOUR AND BASICS

In [2]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")


c:\Users\user\miniconda3\envs\transformers_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\user\miniconda3\envs\transformers_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate d

### TASK 1: TOKEN COUNTING AND DISPLAYING WORD TO TOKEN RATIO

In [3]:
sample_texts = [
    "Hello world!",
    "Natural language processing enables computers to understand text.",
    "Large language models predict the next token based on context.",
    "Deep learning has transformed artificial intelligence research."
]

print("Word Count vs Token Count\n")

for text in sample_texts:
    words = len(text.split())
    tokens = tokenizer.encode(text)
    token_count = len(tokens)

    ratio = words / token_count

    print(f"Text: {text}")
    print(f"Words: {words}")
    print(f"Tokens: {token_count}")
    print(f"Word-to-Token Ratio: {ratio:.2f}")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(tokens)}")
    print("-" * 60)


Word Count vs Token Count

Text: Hello world!
Words: 2
Tokens: 3
Word-to-Token Ratio: 0.67
Tokens: ['Hello', 'Ġworld', '!']
------------------------------------------------------------
Text: Natural language processing enables computers to understand text.
Words: 8
Tokens: 9
Word-to-Token Ratio: 0.89
Tokens: ['Natural', 'Ġlanguage', 'Ġprocessing', 'Ġenables', 'Ġcomputers', 'Ġto', 'Ġunderstand', 'Ġtext', '.']
------------------------------------------------------------
Text: Large language models predict the next token based on context.
Words: 10
Tokens: 11
Word-to-Token Ratio: 0.91
Tokens: ['Large', 'Ġlanguage', 'Ġmodels', 'Ġpredict', 'Ġthe', 'Ġnext', 'Ġtoken', 'Ġbased', 'Ġon', 'Ġcontext', '.']
------------------------------------------------------------
Text: Deep learning has transformed artificial intelligence research.
Words: 7
Tokens: 8
Word-to-Token Ratio: 0.88
Tokens: ['Deep', 'Ġlearning', 'Ġhas', 'Ġtransformed', 'Ġartificial', 'Ġintelligence', 'Ġresearch', '.']
----------------

### TASK 2: TEMPERATURE COMPARISON

In [4]:
# Importing  Text Generation Pipeline
from transformers import pipeline
import warnings

warnings.filterwarnings("ignore")

generator = pipeline(
    "text-generation",
    model="gpt2"
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use cpu


In [5]:
prompt = "The future of renewable energy depends on"     # SAME PROMPT FOR BOTH TEMPERATURES

print("Prompt:")
print(prompt)

print("\nTemperature = 0 (Deterministic)\n")             # TEMPERATURE 0.0

for i in range(3):
    output = generator(
        prompt,
        max_new_tokens=15,
        temperature=0.0,
        do_sample=False
    )

    print(f"Run {i+1}:")
    print(output[0]["generated_text"])
    print()

# HIGHER TEMPERATURE INCREASES RANDOMNESS AND DIVERSITY IN THE OUTPUT, LEADING TO MORE CREATIVE BUT LESS PREDICTABLE RESPONSES.
print("Temperature = 1.0 (Random)\n")                 # TEMPERATURE 1.0

for i in range(3):
    output = generator(
        prompt,
        max_new_tokens=15,
        temperature=1.0,
        do_sample=True
    )

    print(f"Run {i+1}:")
    print(output[0]["generated_text"])
    print()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt:
The future of renewable energy depends on

Temperature = 0 (Deterministic)



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Run 1:
The future of renewable energy depends on the ability of the world's most powerful power companies to meet their renewable energy



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


...

Run 3:
The future of renewable energy depends on the ability of the world's most powerful power companies to meet their renewable energy

Temperature = 1.0 (Random)



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Run 1:
The future of renewable energy depends on a new renewable energy technology that doesn't need coal if coal has enough carbon



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Run 2:
The future of renewable energy depends on a new direction for the world's nations, both in how we use our

Run 3:
The future of renewable energy depends on climate change, with the global potential of a "deconstructed sun for



### TASK 3: STRUCTURED EXTRACTION FROM MESSY TEXT

In [6]:
import json

messy_text = """
Employee Name : Shibam Pal

Reach me at pshibam05@gmail.com

Phone Number -> +91 9831369788

Current Position:
Junior Data Analyst

Expertise includes SQL, Python, Tableau,
Machine Learning, Statistics

Professional Experience:
2 years
"""

extractor = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = f"""
Extract the following information and return ONLY valid JSON.

Schema:

{{
"name": "",
"email": "",
"phone": "",
"position": "",
"skills": [],
"experience_years": null
}}

Text:

{messy_text}

Output only JSON.
"""

result = extractor(
    prompt,
    max_new_tokens=150,
    temperature=0.3,
    do_sample=True
)[0]["generated_text"]

print(result)

Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Extract the following information and return ONLY valid JSON.

Schema:

{
"name": "",
"email": "",
"phone": "",
"position": "",
"skills": [],
"experience_years": null
}

Text:


Employee Name : Shibam Pal

Reach me at pshibam05@gmail.com

Phone Number -> +91 9831369788

Current Position:
Junior Data Analyst

Expertise includes SQL, Python, Tableau,
Machine Learning, Statistics

Professional Experience:
2 years


Output only JSON.

Schema:

{

"name": "",

"email": "",

"phone": "",

"position": "",

"skills": [],

"experience_years": null

}

Text:


Employee Name : Shibam Pal

Reach me at pshibam05@gmail.com

Phone Number -> +91 9831369788

Current Position:

Junior Data Analyst

Expertise includes SQL, Python, Tableau,

Machine Learning, Statistics

Professional Experience:

2 years


Output only JSON.

Schema:

{




### TASK 4: JSON Validation (Graceful Parse Failure)

In [7]:
print("Attempting JSON Parsing...\n")

try:

    start = result.find("{")

    json_text = result[start:]

    parsed = json.loads(json_text)

    print("JSON Parsed Successfully!\n")

    print(json.dumps(parsed, indent=4))

except Exception as e:

    print("JSON Parsing Failed!")

    print("Reason:", e)

    print("\nThe model may generate extra text or malformed JSON.")

    print("Production applications should always validate model output.")

Attempting JSON Parsing...

JSON Parsing Failed!
Reason: Extra data: line 10 column 1 (char 98)

The model may generate extra text or malformed JSON.
Production applications should always validate model output.


### BONUS TASK: Extraction function that retries if parsing fails

In [8]:
import json

def extract_to_dict(text, retries=3):

    prompt = f"""
Extract structured information from the text below.

Return ONLY valid JSON.

Text:
{text}
"""

    for attempt in range(retries):

        output = extractor(
            prompt,
            max_new_tokens=150,
            temperature=0.2,
            do_sample=True
        )[0]["generated_text"]

        try:

            start = output.find("{")

            parsed = json.loads(output[start:])

            print(f"Success on attempt {attempt+1}")

            return parsed

        except Exception:

            print(f"Attempt {attempt+1} failed. Retrying...")

    return {
        "error": "Unable to parse valid JSON after multiple attempts."
    }

# TESTING THE FUNCTION
test_text = """
Candidate:
Shibam Pal

Email : pshibam05@gmail.com

Phone : +91 9831369788

Role : Software Engineer

Skills include Java, Spring Boot, Docker,
Kubernetes, AWS

Experience : 2 years
"""

data = extract_to_dict(test_text)

print(data)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Attempt 1 failed. Retrying...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Attempt 2 failed. Retrying...
Attempt 3 failed. Retrying...
{'error': 'Unable to parse valid JSON after multiple attempts.'}


In [9]:
print("Day 5 complete!")
print("Key insight: temperature = randomness, not quality.")
print("And always validate LLM output -- it WILL break sometimes.")

Day 5 complete!
Key insight: temperature = randomness, not quality.
And always validate LLM output -- it WILL break sometimes.
